# Ferry tickets — EDA & modeling

This notebook documents exploratory analysis and points to the canonical training script.

- Raw data: `../data/ferry_tickets.csv`
- Training (all models, metrics, figures): `python ../scripts/train_all.py` from project root
- Shared preprocessing: `../backend/services/preprocessor.py`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from backend.services.preprocessor import load_raw_ferry_csv, reindex_to_15min_grid, winsorize_iqr_by_hour, build_feature_matrix

data_path = ROOT / "data" / "ferry_tickets.csv"
raw = load_raw_ferry_csv(data_path)
gridded, missing = reindex_to_15min_grid(raw)
clean = winsorize_iqr_by_hour(gridded)
full, feat_cols, tgt_cols = build_feature_matrix(clean)
print("Rows:", len(raw), "Date range:", raw.index.min(), raw.index.max())
print("Missing 15m slots (pre-interpolation):", missing)
print("Feature columns:", len(feat_cols), "Target columns:", tgt_cols)

In [ ]:
desc = clean.describe()
print(desc)
weekend = clean.index.dayofweek >= 5
print("Weekend mean sales:", clean.loc[weekend, "sales"].mean())
print("Weekday mean sales:", clean.loc[~weekend, "sales"].mean())

## Train all models

Run from a terminal in the project root:

```bash
export PYTHONPATH=.
python scripts/train_all.py
```

Optional: `FERRY_TRAIN_MAX_ROWS=40000` for faster iteration, `SKIP_SARIMA=1` if `pmdarima` fails to import.